# Practical Human Resource Allocation in Software Projects using Genetic Algorithm

## Introduction

Efficient allocation of human resources is one of the most critical and challenging problems in software project management. Software projects are constrained not only by task dependencies and deadlines, but also by human factors such as developer skills, experience levels, multitasking limitations, and knowledge continuity between tasks. Poor allocation decisions often lead to delays, excessive task switching, productivity loss, and increased project risk.

This project is a **replication and experimental reimplementation** of the research work entitled:

> **“Practical Human Resource Allocation in Software Projects Using Genetic Algorithms”**  
> Jihun Park, Dongwon Seo, Gwangui Hong, Donghwan Shin, Jimin Hwa, and Doo-Hwan Bae

The main objective of this project is to **develop a replica of the original approach proposed in the article and test it through our own implementation**. We do not aim to improve the original model, but rather to understand it better by rebuilding it and observing its behavior through experiments.

In this project, the software development process is modeled as an optimization problem. Each task has a workload and may depend on other tasks. Each developer has different skills and a specific experience level. A **Genetic Algorithm (GA)** is used to search for good task–developer assignments among many possible solutions.

To evaluate each solution, we simulate the execution of the project step by step. The simulator takes into account:
- Task dependencies
- Developer skills and levels
- The negative effect of multitasking
- The loss of efficiency when too many developers work on the same task

By running different experiments, we compare multi-objective optimization (CaseAll) with time-only optimization (CaseTime). The results show that considering human factors leads to more realistic and balanced schedules, which confirms the main ideas presented in the original article.


## Modeling the Input Data Sets

The input data represents a software project tasks and the people working on it. In our code, this is done using two main elements: **developers** and **tasks**.

### Developers

Each developer is modeled with three attributes:
- **ID**: a unique identifier (e.g. `d1`, `d2`)
- **Abilities**: a dictionary that defines the developer’s competence for each task type  
  (`analysis`, `design`, `implementation`, `testing`)
- **Level**: the experience level of the developer  
  (`junior`, `engineer`, `manager`, `director`)

The developer’s level is used to scale their abilities through a level factor. This reflects the idea that more experienced developers are generally more efficient, even when performing the same type of task.

### Tasks

Each task is defined by:
- **ID**: a unique task name (e.g. `t1`, `t2`)
- **Type**: the nature of the task (`analysis`, `design`, `implementation`, `testing`)
- **Effort**: the total amount of work required to complete the task
- **Dependencies**: a list of tasks that must be completed before this task can start

This structure allows us to model realistic project workflows with dependencies between tasks.

### Input Data Sets

We created **three different input data sets** to test the algorithm under different conditions:

- **Set 1**:  
  A small project with a limited number of developers and tasks.  
  This set is mainly used to validate the accuracy of the model and the algorithm.
  
- **Set 2**:  
  The same task structure as Set 1, but with additional developers.  
  This allows us to study how increasing team size affects project duration and multitasking.

- **Set 3**: 
  A larger project with more tasks, more dependencies, and more developers.  
This set is used to observe how the algorithm behaves when the project becomes bigger and more complex.

In [10]:
import random, math
from collections import deque

# Fixer la graine (seed) aléatoire pour garantir la reproductibilité :
# à chaque exécution, les tirages random (population initiale, mutation, tournoi, crossover…) seront identiques.
random.seed(42)

# Facteur de seniorité pour prendre en compte l'experience de developpeur en compte.
LEVEL_FACTOR={"junior":.8,"engineer":1.0,"manager":1.1,"director":1.2}

# Structures de données 
# Un développeur = (id, compétences par type de tâche, niveau/seniorité).
class Developer: 
    def __init__(s,id,ab,lv): 
        s.id=id; s.abilities=ab; s.level=lv

# Une tâche = (id, type, effort à réaliser, liste de prérequis).
class Task:
    def __init__(s,id,typ,eff,pre): 
        s.id=id; s.typ=typ; s.effort=eff; s.prereq=list(pre)

### Genetic Algorithm:

#### 1. Chromosome Representation
A chromosome represents one possible project plan:

{"t1": ["d1", "d3"], "t2": ["d2"], ...}
Each chromosome fully defines how developers are allocated to all tasks.

#### 2. Project Simulation (`simulate`)
The function `simulate()` evaluates a chromosome by simulating the project execution step by step:
- A task can start only when all its prerequisites are completed
- Developers may work on multiple tasks at the same time
- Developer efficiency is reduced when multitasking
- A coordination penalty is applied when many developers work on the same task

The simulation returns the total project duration and developer workload estimates.

#### 3. Fitness Evaluation (`scores`)
Each solution is evaluated using four criteria:
- **CM (Cost Metric)**: compares project duration to an optimistic lower bound
- **CE (Concentration Efficiency)**: penalizes multitasking
- **CC (Continuity Consideration)**: rewards using the same developers across dependent tasks
- **BA (Balance Allocation)**: measures how evenly work is distributed across staff levels

The final fitness is a weighted sum of these four metrics.

#### Genetic Algorithm Operators
The main GA logic is implemented in the function `run_ga_case()`.
- **Initialization (`init_pop`)**: generates a random population of solutions
- **Selection (`select`)**: chooses parents using tournament selection
- **Crossover (`crossover`)**: mixes assignments from two parents
- **Mutation (`mutate`)**: randomly modifies chromosomes to maintain diversity
- **Elitism**: preserves the best solutions across generations

---

#### Pseudocode

```text
Initialize population with random chromosomes
Evaluate fitness of each chromosome

FOR each generation DO
    Keep best individuals (elitism)

    WHILE population is not full DO
        Select two parents
        Apply crossover
        Apply mutation
        Evaluate new chromosome
        Add chromosome to new population
    END WHILE
    Update best solution found
END FOR

Return best chromosome

In [11]:
# Fonction principale qui permet d'implementer l'algorithme genetique
def ga(tasks, devs, W, pop=20, gens=50, mut=0.03, tourn=3, elit=2, kmax=3, verbose=False):
    # Indexation (TaskID->index, DevID->index) pour accélérer les calculs en tableaux.
    # on a créé des listes d'index pour les taches tidx et developpeurs didx car c'est plus rapide de parcourir les listes avec des indexes que des strings
    tids=list(tasks); tidx={t:i for i,t in enumerate(tids)}; nT=len(tids)
    dids=[d.id for d in devs]; didx={d:i for i,d in enumerate(dids)}; nD=len(dids)

    # Infos utiles pour le calcul de score BA
    dev_level={d.id:d.level for d in devs}; #On construit un dictionnaire qui associe chaque développeur à son niveau, id:level
    levels=sorted(set(d.level for d in devs)) #Elle extrait tous les niveaux de développeurs, sans doublons, et les trie.
                                              # set(...) Supprime les doublons, sorted(...) Trie le résultat
    # Construction du graphe de dépendances des tâches et d'effort.
    effort=[0.0]*nT; #pour initialiser le tableau, c'est une liste de longueur nT remplie de zéros 0.0
    prereq=[[] for _ in range(nT)]; adj=[[] for _ in range(nT)] #des tableaux qui contiennent des listes vides (nombre nT), une pour chaque tache
    for tid,t in tasks.items():      #pour chaque tache et son indice 
        i=tidx[tid]                            # index de la tâche tid
        effort[i]=float(t.effort)              # ajouter l'effort de la tache i au tableau des efforts 
        prereq[i]=[tidx[p] for p in t.prereq]  # ajouter les prérequis (dependances) de la tache i au tableau des prérequis
        for p in prereq[i]: adj[p].append(i)   # adj[p] contient les successeurs de p, si p est un prérequis de i, alors i est un successeur de p.

    # Matrice speed[dev_index][task_index] = vitesse effective (compétence * LEVEL_FACTOR)
    # c'est une matrice 2D et représente la vitesse réelle de travail du développeur d sur la tâche t
    speed=[[0.0]*nT for _ in range(nD)]
    for d in devs:
        di=didx[d.id]; lf=LEVEL_FACTOR.get(d.level,1.0)
        for tid,t in tasks.items(): #tasks.items() retourne les deux choses en même temps, tid l’ID de la tâche ("t1") et t l’objet Task correspondant.
            speed[di][tidx[tid]]=d.abilities.get(t.typ,0.0)*lf #On récupère la compétence du dév (dans abilities) d pour la tache t typ * lf

    # Pénalité de coordination quand k développeurs travaillent sur la même tâche (rendements décroissants).
    coord=lambda k: 1.0 if k<=1 else max(0.4,1.0-0.12*(k-1)) #lambda permet de créer une fonction très courte sans utilisé def coord(k)
    #nomFunc= lambda argument: expression  %l'argument est k 

    # Nettoyage d’un chromosome (allocation tâches->devs) :
    # - supprime doublons, - supprime IDs invalides, - limite à kmax devs/tâche, - garantit ≥1 dev/tâche.
    def sanitize(ch):
        for tid in tids:
            seen=set(); out=[] #seen est un set (n'accepte pas de doublons) -> pour vérifier, out est la liste qui stocke le resultat final
            for d in ch.get(tid,[]): #Si tid existe dans le chromosome ch --> retourne la liste des devs pour cette tache
                if d in didx and d not in seen: #si d existe déja en didx (pour chaque d de ch), et n'est pas encore ajouté à seen
                    seen.add(d); out.append(d) #on ajoute ce d,.add() pour les sets (ensembles) et .append() pour les listes 
            ch[tid]=out[:kmax] if out else [random.choice(dids)] #forme: A if condition else B, si condition est vraie -> on prend A, sinon → on prend B
        return ch                          #random.choice(dids) choisit un développeur au hasard dans la liste dids
                                           # if out : si la liste out n’est pas vide, out[:kmax] : on garde seulement les kmax premiers éléments de la liste out
    
    # Borne optimiste minTS (chemin critique) : ici on calcule le Critical Time Span ou la durée de chemin critique
    # on suppose que chaque tâche est faite par le meilleur développeur possible pour son type.
    def minTS():
        best=[] #duree min pour la tache i
        for ti in range(nT):
            best_sp=max(speed[di][ti] for di in range(nD))  #on prend la vitesse max pour chaque tache ti,en calculant tout les combinaisons possible
            best.append(effort[ti]/best_sp if best_sp>0 else float("inf")) #on calcule la duree (effort/vitesse), si y a pas de best speed>0 on prend l'infini
        indeg=[len(prereq[i]) for i in range(nT)]   #indeg[i] = nombre de dépendances restantes (nombre de prérequis) pour i
        q=deque([i for i in range(nT) if indeg[i]==0])  # q : liste des tâches sans prérequis =taches prêts à etre executées
        # deque: file d'attente à deux extrémités (double-ended queue), on l'utilise pour enlever un élément au début ou l'ajouter à la fin
        dp=best[:]  # dp[i] = durée max pour atteindre i (chemin le plus long), best[:] = copie de la liste best, on va l’augmenter selon les dépendances.
        seen=0
        while q:  #tant que q n'est pas vide
            u=q.popleft(); seen+=1 #On enlève la première tâche prête de q (principe Fifo) et on le met en u
            for v in adj[u]: #on va traiter v qui est la succésseur de u
                dp[v]=max(dp[v], dp[u]+best[v]) #son durée est la plus grande entre sa durée initiale et celle de dp[u]+best[v]
                indeg[v]-=1                       #on decremente les prerequis de v car on suppose que u est déja realisé
                if indeg[v]==0: q.append(v)       #Si v n’a plus de prérequis, elle devient exécutable et on l’ajoute à la file q
        return max(dp) if seen==nT else min(best)*nT   #Si toutes les tâches ont été traitées (seen == nT) on retourne max(dp) sinon...

    MIN_TS=minTS()

    # Simulateur discret : exécute les tâches “enabled” à chaque tick, applique multitâche + coordination,
    # renvoie (finish_time, dev_stats).
    def simulate(ch, max_ticks=20000): #max_ticks est une limite de sécurité, si la simulation dure trop longtemps, on stoppe.
        rem=effort[:]             # effort restant, rem[i]= effort restant pour la tâche i
        done=[False]*nT          # une liste de taille nT initialisé par des False, done[i]=True quand la tâche i termine.
        need=[len(prereq[i]) for i in range(nT)]         # nb de prérequis restants pour la tache i
        enabled=[i for i in range(nT) if need[i]==0 and rem[i]>0]  #tâches i prêtes à être exécutées, si y'a aucun prérequis restant et il reste du travail à fair

        load=[0]*nD                # load[dev]=nb de tâches sur lesquelles ce dev travaille dans le tick courant
        tw=[0]*nD; tm=[0]*nD      # tw[dev]= nombre de ticks où le dev a travaillé au moins sur 1 tâche (“ticks_with_work”)
                                  # tm[dev]= nombre de ticks où le dev a travaillé sur plus d’une tâche (“ticks_with_multitask”)
        t=0; done_cnt=0    #t=temps actuel (tick actuel), done_cnt = combien de tâches sont terminées.
        while done_cnt<nT and t<max_ticks:  #tant qu'on a pas terminé tout les taches et qu'il reste de temps encore
            # Cas où aucune tâche n’est exécutable
            if not enabled: t+=1; continue   # si enabled est une liste vide ( il reste plus de tache prete), On avance le temps d’un tick

            # Calcul de la charge par dev (multitâche) sur ce tick.
            for di in range(nD): load[di]=0  #la charge initiale de chaque dev est 0
            
            for ti in enabled:  # si ti est prete
                for d in ch[tids[ti]]: # si d est dans la liste des devs affectés à ti
                    di=didx.get(d)     # di retourne l'ID de d si d existe, sinon .get() renvoie None si d n’existe pas
                    if di is not None: load[di]+=1   #ici on s'assure que le dév existe bien car il peut etre None

            # CE : nombre de ticks où le dev travaille / multitâche.
            for di in range(nD):
                if load[di]>0:   #Si le dev travaille sur au moins une tâche en ce tick
                    tw[di]+=1    #On compte un tick de travail pour ce dev
                    if load[di]>1: tm[di]+=1  #Si le dev a plus d’une tâche en même temps -> multitâche tm+1

            # Avancement des tâches enabled.
            finished=[]  #l'ensemble de tâches qui terminent pendant ce tick de temps, On l’utilise ensuite pour débloquer leurs successeurs
            for ti in enabled:  #si ti est prete
                raw=0.0; cnt=0  #cnt= nb de devs affectés à cette tâche, raw= contributions réelle (en vitesse) des devs sur cette tâche pour ce tick
                for d in ch[tids[ti]]: #si d est affecté à ti dans la solution de GA
                    di=didx.get(d)      #on récupère son id
                    if di is None: continue #si di=None (n'existe pas) on ignore
                    raw += speed[di][ti]/(load[di] if load[di] else 1)  # partage de vitesse si multitâche (si load[di]= 0 on divise par 1)
                    # un dev en multitâche travaille moins vite sur chaque tâche --> donc la vitesse sera partagee
                    cnt += 1  #on incremente le nb de devs travaillant sur la tache ti
                if cnt:   #si cnt> 0 signifie : la tâche a reçu du travail ce tick
                    rem[ti]-=raw*coord(cnt)    # on reduit l’effort restant en appliquant la pénalité de coordination
                    if rem[ti]<=0 and not done[ti]:   #si la tache l'effort restant de ti=0 et elle n'est pas marqué comme fini(done=true)
                        done[ti]=True; finished.append(ti); done_cnt+=1 # marquer comme finie (finished = terminée pendant ce tick,done_cnt= compteur global) 

            # Débloquer les successeurs quand des tâches finissent.
            if finished:  #contient les tâches qui viennent de terminer dans ce tick, si la liste est vide (aucune tache n'a fini dans ce tick) y a rien à debloquer
                fin=set(finished) # le set est pour faire un test rapide
                enabled=[x for x in enabled if x not in fin] #on retire les taches déja qui vient de terminer (x en fin)
                for u in finished:   #u a fini
                    for v in adj[u]: # v est son successeur
                        need[v]-=1    #on enleve cette tache de nombre de prerequis pour v
                        if need[v]==0 and (not done[v]) and rem[v]>0:  # si il reste aucun prerequis pour v, et v n'est pas marquée done, et il reste de travail à faire en v
                            enabled.append(v)  # v est ajoutée aux taches pretes pour etre executée apres
            t+=1    # on passe au tick suivante (l'unité de temps +1)

        return t, {dids[i]:{"ticks_with_work":tw[i],"ticks_with_multitask":tm[i]} for i in range(nD)} #ça retourne t, dev_stats (qui est un dictionnaire de dev)
        # t est durée totale simulée (finish time), tw et tm permet de calculer le temps où le dev a travaillé/ en multitache

    # Calcule les scores CM, CE, CC, BA + fitness globale
    def score(ch):
        ch=sanitize(ch)
        finish, ds = simulate(ch) #on lance la simulation ->(t,dev_stats) on met t en finish et dev_stats en ds

        # CM (makespan) = minTS / max(finish_time, minTS)
        cm=0.0 if MIN_TS<=0 or math.isinf(MIN_TS) else MIN_TS/max(finish,MIN_TS)

        # CE (concentration) : on fait la somme de [1-(temps multitâche/ temps travaillé) pour chaque dev, et on divise par nb de devs
        ce=sum(1.0 if ds[d]["ticks_with_work"]==0 else max(0.0,1.0-ds[d]["ticks_with_multitask"]/ds[d]["ticks_with_work"]) #si le dev n'a pas travaillé tw==0 on met 1 dans son score, le max est pour eviter d'avoir une val negative
               for d in dids)/nD  #si temps_multitache==0 le score CE=1 (le meilleur cas), si t_multitâche==t_travaillé le score CE=0 (le pire cas)

        # CC (continuity) = nombre total de relations de dépendance/ nombre de dépendances respectant la continuité
        total=ok=0 #total: nombre total de relations de dépendance, ok: nombre de dépendances respectant la continuité
        for tid,t in tasks.items():  #.items() permet de retourner à la fois l'objet task t et son id tid
            for p in t.prereq:   #si une relation de dependance trouvé
                total+=1         #on compte cette dependace
                ok += 1 if (set(ch[tid]) & set(ch[p])) else 0  # & est l'intersection de deux ensembles,s'il est vide ok=0 (aucun dev en commun)
                #ch[tid]= liste des devs affectés à tid, ch[p]= liste des devs affectés à p
        cc=ok/total if total else 1.0

        # BA (balance) : entropie modelisant la distribution des efforts par niveau de staff.
        ba_vals=[]
        for lvl in levels:  #pour chaque niveau de staff
            #[sum() compte combien de devs de niveau lvl travaille sur tid]*l'effort de la tache tid
            nums=[sum(1 for d in ch[tid] if dev_level.get(d)==lvl)*tasks[tid].effort for tid in tids] 
            denom=sum(nums) #fait la somme sur toute les taches, calcule l'effort total réalisé par ce niveau de staff 
            if denom==0: ba_vals.append(1.0); continue #si ce niveau ne travail pas, on lui donne 1 pour ne pas penaliser un niveau absent
            #nums[]: effort de ce niveau par tâche, denom: effort total de ce niveau
            #ps=nums/denom est la distribution de probabilité dont la somme=1 (%d'effort par tache)
            ps=[x/denom for x in nums if x>0]
            ba_vals.append(-sum(p*(math.log(p)/math.log(nT)) for p in ps))  #formule d'entropie de Shannon: BA=-sum(p*log_nT(p))
        ba=sum(ba_vals)/len(ba_vals) if ba_vals else 0.0   #BA final= moyenne des entropies sur tous les niveaux de staff

        # Fitness finale (pondérations/poids W)
        # w est un dict de poids: W = {"cm":0.25, "ce":0.25, "cc":0.25, "ba":0.25} en CaseAll
        fit=W["cm"]*cm+W["ce"]*ce+W["cc"]*cc+W["ba"]*ba  #chaque score est multiplié par son importance
        return fit, {"cm":cm,"ce":ce,"cc":cc,"ba":ba,"sim":{"finish_time":finish,"dev_stats":ds}}

    # Initialisation aléatoire de la population (chromosomes).
    # Population = liste (nombre pop d'individus, Chaque solution = dictionnaire
    #random.randint(1, kmax): Retourne un entier aléatoire entre 1 et kmax (inclus), random.sample(liste, k): Choisit k éléments différents au hasard dans liste
    init=lambda: [sanitize({tid: random.sample(dids, random.randint(1,kmax)) for tid in tids}) for _ in range(pop)]

    # Sélection tournoi : on choisit le meilleur parmi “tourn” individus tirés au hasard.
    def pick(P, F):       #pick(P,F): P est population, F est la fitness associée à chaque chromosome
        best_index = max(random.sample(range(pop), tourn), key=lambda i: F[i])
                     #max(liste, key=fonction) max parcourt la liste et compare les éléments selon la valeur retournée par fonction (F[i] ici), liste retourne des indices i
        return {tid: P[best_index][tid][:] for tid in tids}  # Retourner une COPIE du chromosome sélectionné

    # Crossover uniforme (avec 50% de chance): pour chaque tâche, on copie le gène du parent1 ou du parent2.
    cross=lambda a,b: {tid:(a[tid][:] if random.random()<0.5 else b[tid][:]) for tid in tids}   #random.random(): nombre réel aléatoire entre 0 et 1
    # dans un chromosome enfant, pour chaque tache tid soit on copie le gène (les devs) de parent a soit de b avec probabilité 50 %
   
    # Mutation : add/remove/replace un développeur sur certaines tâches, puis sanitize.
    def mutate(ch):
        for tid in tids:
            if random.random()<mut:  #random.random() est un chiffre entre 0 et 1, fait la mutation de manière aleatoire
                op=random.choice(["add","remove","replace"])   #on choisit aléatoirement une opération parmi 3
                if op=="add" and len(ch[tid])<kmax:            #On n’ajoute que si on n’a pas dépassé kmax (max devs par tâche)
                    cand=[d for d in dids if d not in ch[tid]] #on construit une liste des candidats devs qui ne sont pas encore assignés à la tache tid
                    if cand: ch[tid].append(random.choice(cand)) #si la liste n’est pas vide, on ajoute un dev aléatoire parmi les candidats
                elif op=="remove" and len(ch[tid])>1:            #On retire un dev seulement si on en a plus d’un
                    ch[tid].pop(random.randrange(len(ch[tid])))  #pop(i) supprime l’élément i
                elif op=="replace":
                    i=random.randrange(len(ch[tid]))            #random.randrange(n) donne un index aléatoire entre 0 et n-1 (nombre de devs assignés à tid)
                    cand=[d for d in dids if d not in ch[tid]]
                    if cand: ch[tid][i]=random.choice(cand)     #on remplace l’élément ch[tid][i] par un autre dev au hasard
        return sanitize(ch)

    # Boucle GA
    P=init()           #on cree la population de départ
    F,D=zip(*(score(ch) for ch in P)); #score(ch) renvoie pour chaque chromosome la liste (fit,details)
                                       #zip(*..) sert à separer le tuple en 2 liste: une pour F et une pour D
    F=list(F); D=list(D)   #on convertit F et D en liste pour pouvoir les modifier/indexer
    b=max(range(pop), key=lambda i:F[i]); best=(P[b],F[b],D[b]) #P[b] est le meilleur individu de pop initial situé à l'indice b, on le stock en best pour demarrer l'algorithme
                                                            
    for g in range(gens):       #pour chaque generation g :
        # Elitisme : garder les meilleurs individus sans modification.
        r=sorted(range(pop), key=lambda i:F[i], reverse=True)  #reverse=True sorting en ordre décroissant
        newP=[P[i] for i in r[:elit]]     #r[:elit]: les indices des elit (2 premiers car elit=2) meilleurs individus
        newF=[F[i] for i in r[:elit]]     # On copie les meilleurs inds dans la nouvelle population newP, newF, newD
        newD=[D[i] for i in r[:elit]]

        # Générer le reste par sélection + crossover + mutation.
        while len(newP)<pop:     #tant que la taille de newP est inf à pop=50
            c=mutate(cross(pick(P,F), pick(P,F)))  #avec pick on prends deux parents aleatoire, on applique le crossover-->enfant-->mutation --> =C
            f,d=score(c)   #c est le nouveau enfant créé, on lui applique score() pour lui évaluer
            newP.append(c); newF.append(f); newD.append(d)   #on ajoute le nouveau ind à newP

        P,F,D=newP,newF,newD                   #Remplacer l’ancienne population par la nouvelle
        b=max(range(pop), key=lambda i:F[i])   #on cherche le meilleur nouveau individu b
        if F[b]>best[1]: best=(P[b],F[b],D[b]) #si la fitness de b actuel est mieux que le best precedent, on reprend celui-ci comme best 
                                               # best[1] retourne la fitness F[b]
        if verbose: print(f"Gen {g+1}/{gens} best fitness: {best[1]:.4f}")  #Si verbose=True, on affiche les détails pendant l’exécution

    return best
# Somme globale des ticks de multitâche pour l'affichage 
multitask_sum=lambda det: sum(v["ticks_with_multitask"] for v in det["sim"]["dev_stats"].values())

### Function to print the Output

In [12]:
def print_results(chromA, fitA, detA, chromT, fitT, detT):
    timeA, multiA = detA["sim"]["finish_time"], multitask_sum(detA)
    timeT, multiT = detT["sim"]["finish_time"], multitask_sum(detT)

    print("=== COMPARISON TABLE ===")
    print(f"{'Set':<6} | {'Time(All)':>10} | {'Time(Time)':>10} | {'Multitask(All)':>15} | {'Multitask(Time)':>15}")
    print(f"{'Value':<6} | {timeA:>10.2f} | {timeT:>10.2f} | {multiA:>15.2f} | {multiT:>15.2f}")
    # CaseAll : pondérations égales
    print("\n=== BEST SOLUTION (CaseAll) ===")
    print("Best fitness:", round(fitA, 6))
    print("Components:", {k: round(v, 6) for k, v in detA.items() if k in ["cm","ce","cc","ba"]})
    print("Finish time (sim):", detA["sim"]["finish_time"])
    print("Assignments (task -> devs):")
    for tid in sorted(chromA, key=lambda x: int(x[1:])):
        print(f"  {tid} -> {chromA[tid]}")

    # CaseTime : optimiser uniquement la durée (CM)
    print("\n=== BEST SOLUTION (CaseTime) ===")
    print("Best fitness:", round(fitT, 6))
    print("Finish time (sim):", detT["sim"]["finish_time"])
    print("Assignments (task -> devs):")
    for tid in sorted(chromT, key=lambda x: int(x[1:])):
        print(f"  {tid} -> {chromT[tid]}")

## Test 1 : Data Set 1

# Developers 

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#2c3e50; color:white;">
      <th style="padding:8px; border:1px solid #ddd;">Developer</th>
      <th style="padding:8px; border:1px solid #ddd;">Role</th>
      <th style="padding:8px; border:1px solid #ddd;">Analysis</th>
      <th style="padding:8px; border:1px solid #ddd;">Design</th>
      <th style="padding:8px; border:1px solid #ddd;">Implementation</th>
      <th style="padding:8px; border:1px solid #ddd;">Testing</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#ecf0f1;">
      <td style="padding:8px; border:1px solid #ddd;">d1</td>
      <td style="padding:8px; border:1px solid #ddd; color:#c0392b;"><b>Director</b></td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td style="padding:8px; border:1px solid #ddd;">d2</td>
      <td style="padding:8px; border:1px solid #ddd; color:#c0392b;"><b>Director</b></td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td style="padding:8px; border:1px solid #ddd;">d3</td>
      <td style="padding:8px; border:1px solid #ddd; color:#2980b9;"><b>Manager</b></td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td style="padding:8px; border:1px solid #ddd;">d4</td>
      <td style="padding:8px; border:1px solid #ddd; color:#2980b9;"><b>Manager</b></td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td style="padding:8px; border:1px solid #ddd;">d5</td>
      <td style="padding:8px; border:1px solid #ddd; color:#27ae60;"><b>Engineer</b></td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td style="padding:8px; border:1px solid #ddd;">d6</td>
      <td style="padding:8px; border:1px solid #ddd; color:#27ae60;"><b>Engineer</b></td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td style="padding:8px; border:1px solid #ddd;">d7</td>
      <td style="padding:8px; border:1px solid #ddd; color:#27ae60;"><b>Engineer</b></td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
    </tr>
  </tbody>
</table>


# Tasks

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#34495e; color:white;">
      <th style="padding:8px; border:1px solid #ddd;">Task</th>
      <th style="padding:8px; border:1px solid #ddd;">Type</th>
      <th style="padding:8px; border:1px solid #ddd;">Effort (h)</th>
      <th style="padding:8px; border:1px solid #ddd;">Dependencies</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#fdebd0;">
      <td style="padding:8px; border:1px solid #ddd;">t1</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Analysis</b></td>
      <td style="padding:8px; border:1px solid #ddd;">400</td>
      <td style="padding:8px; border:1px solid #ddd;">—</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td style="padding:8px; border:1px solid #ddd;">t2</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Design</b></td>
      <td style="padding:8px; border:1px solid #ddd;">320</td>
      <td style="padding:8px; border:1px solid #ddd;">t1</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td style="padding:8px; border:1px solid #ddd;">t3</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Design</b></td>
      <td style="padding:8px; border:1px solid #ddd;">240</td>
      <td style="padding:8px; border:1px solid #ddd;">t1</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td style="padding:8px; border:1px solid #ddd;">t4</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Design</b></td>
      <td style="padding:8px; border:1px solid #ddd;">240</td>
      <td style="padding:8px; border:1px solid #ddd;">t1</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td style="padding:8px; border:1px solid #ddd;">t5</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Implementation</b></td>
      <td style="padding:8px; border:1px solid #ddd;">240</td>
      <td style="padding:8px; border:1px solid #ddd;">t2</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td style="padding:8px; border:1px solid #ddd;">t6</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Implementation</b></td>
      <td style="padding:8px; border:1px solid #ddd;">600</td>
      <td style="padding:8px; border:1px solid #ddd;">t3</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td style="padding:8px; border:1px solid #ddd;">t7</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Implementation</b></td>
      <td style="padding:8px; border:1px solid #ddd;">160</td>
      <td style="padding:8px; border:1px solid #ddd;">t4</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td style="padding:8px; border:1px solid #ddd;">t8</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Testing</b></td>
      <td style="padding:8px; border:1px solid #ddd;">100</td>
      <td style="padding:8px; border:1px solid #ddd;">t5</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td style="padding:8px; border:1px solid #ddd;">t9</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Testing</b></td>
      <td style="padding:8px; border:1px solid #ddd;">80</td>
      <td style="padding:8px; border:1px solid #ddd;">t6</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td style="padding:8px; border:1px solid #ddd;">t10</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Testing</b></td>
      <td style="padding:8px; border:1px solid #ddd;">70</td>
      <td style="padding:8px; border:1px solid #ddd;">t7</td>
    </tr>
    <tr style="background-color:#f5b7b1;">
      <td style="padding:8px; border:1px solid #ddd;">t11</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Testing</b></td>
      <td style="padding:8px; border:1px solid #ddd;">80</td>
      <td style="padding:8px; border:1px solid #ddd;">t8, t9, t10</td>
    </tr>
  </tbody>
</table>


In [13]:
# Données Set1
def generate_set1_data():
    # Liste des développeurs avec leurs vitesses par type et leur niveau.
    devs=[Developer("d1",{"analysis":1.25,"design":1.0,"implementation":1.25,"testing":1.25},"director"),
          Developer("d2",{"analysis":1.25,"design":1.0,"implementation":1.25,"testing":0.75},"director"),
          Developer("d3",{"analysis":0.75,"design":0.75,"implementation":1.0,"testing":1.0},"manager"),
          Developer("d4",{"analysis":0.75,"design":1.0,"implementation":1.0,"testing":0.75},"manager"),
          Developer("d5",{"analysis":1.0,"design":0.75,"implementation":0.75,"testing":1.0},"engineer"),
          Developer("d6",{"analysis":0.75,"design":1.0,"implementation":0.75,"testing":0.75},"engineer"),
          Developer("d7",{"analysis":1.0,"design":0.75,"implementation":1.0,"testing":0.75},"engineer")]
   
    # Dictionnaire des tâches : effort + dépendances (prérequis).
    tasks={"t1":Task("t1","analysis",400,[]),
           "t2":Task("t2","design",320,["t1"]),
           "t3":Task("t3","design",240,["t1"]),
           "t4":Task("t4","design",240,["t1"]),
           "t5":Task("t5","implementation",240,["t2"]),
           "t6":Task("t6","implementation",600,["t3"]),
           "t7":Task("t7","implementation",160,["t4"]),
           "t8":Task("t8","testing",100,["t5"]),
           "t9":Task("t9","testing",80,["t6"]),
           "t10":Task("t10","testing",70,["t7"]),
           "t11":Task("t11","testing",80,["t8","t9","t10"])}
    return tasks, devs

### Run the Genetic Algorithm for this data set :

In [14]:
if __name__ == "__main__":
    tasks, devs = generate_set1_data()

    # CaseAll : pondérations égales
    chromA, fitA, detA = ga(tasks, devs, {"cm":.25,"ce":.25,"cc":.25,"ba":.25})

    # CaseTime : optimiser uniquement la durée (CM)
    chromT, fitT, detT = ga(tasks, devs, {"cm":1.0,"ce":0.0,"cc":0.0,"ba":0.0})

    print_results(chromA, fitA, detA, chromT, fitT, detT)

=== COMPARISON TABLE ===
Set    |  Time(All) | Time(Time) |  Multitask(All) | Multitask(Time)
Value  |     824.00 |     833.00 |          111.00 |          741.00

=== BEST SOLUTION (CaseAll) ===
Best fitness: 0.948845
Components: {'cm': 1.0, 'ce': 0.979978, 'cc': 1.0, 'ba': 0.8154}
Finish time (sim): 824
Assignments (task -> devs):
  t1 -> ['d3', 'd6', 'd2']
  t2 -> ['d1', 'd5', 'd6']
  t3 -> ['d3']
  t4 -> ['d2', 'd7', 'd4']
  t5 -> ['d1', 'd4']
  t6 -> ['d5', 'd3', 'd1']
  t7 -> ['d7', 'd2', 'd3']
  t8 -> ['d5', 'd1', 'd2']
  t9 -> ['d1', 'd5', 'd4']
  t10 -> ['d5', 'd6', 'd3']
  t11 -> ['d7', 'd3', 'd5']

=== BEST SOLUTION (CaseTime) ===
Best fitness: 1.0
Finish time (sim): 833
Assignments (task -> devs):
  t1 -> ['d1', 'd6', 'd2']
  t2 -> ['d5', 'd4', 'd1']
  t3 -> ['d1', 'd3']
  t4 -> ['d1', 'd2', 'd6']
  t5 -> ['d1', 'd4']
  t6 -> ['d7', 'd3']
  t7 -> ['d3', 'd4']
  t8 -> ['d6', 'd5']
  t9 -> ['d1', 'd6']
  t10 -> ['d1', 'd3', 'd7']
  t11 -> ['d5', 'd6', 'd7']


## Interpretation of the Results

### 1. Comparison Between CaseAll and CaseTime

The comparison table summarizes the performance of the genetic algorithm on **Set1** under two different optimization strategies:

- **CaseAll**: all objective components are weighted equally  
  (CM, CE, CC, BA each have a weight of 0.25).
- **CaseTime**: only the project duration (CM) is optimized  
  (CM = 1.0, all other weights = 0).

| Metric | CaseAll | CaseTime |
|------|---------|----------|
| Finish time | 824 | 833 |
| Multitasking | 111 | 741 |

**Observations:**
- CaseAll finishes slightly faster than CaseTime (824 vs 833).
- CaseAll strongly reduces multitasking compared to CaseTime.
- CaseTime allows much more multitasking because it only tries to minimize time and ignores developer concentration.

---

### 2. Best Solution for CaseAll

The best solution found with **CaseAll** reaches a **fitness of 0.948845**, which is very close to the theoretical maximum of 1.

### 3. Best Solution for CaseTime

For **CaseTime**, the fitness is **1.0**, which is expected because the objective only considers project duration.

This confirms the importance of using multi-objective optimization rather than optimizing project duration alone.


## Test 2 : Data Set 2


# Developers

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#1f2d3d; color:white;">
      <th style="padding:8px; border:1px solid #ccc;">Developer</th>
      <th style="padding:8px; border:1px solid #ccc;">Role</th>
      <th style="padding:8px; border:1px solid #ccc;">Analysis</th>
      <th style="padding:8px; border:1px solid #ccc;">Design</th>
      <th style="padding:8px; border:1px solid #ccc;">Implementation</th>
      <th style="padding:8px; border:1px solid #ccc;">Testing</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#ecf0f1;">
      <td>d1</td><td style="color:#c0392b;"><b>Director</b></td>
      <td>1.25</td><td>1.00</td><td>1.25</td><td>1.25</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d2</td><td style="color:#c0392b;"><b>Director</b></td>
      <td>1.25</td><td>1.00</td><td>1.25</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d3</td><td style="color:#2980b9;"><b>Manager</b></td>
      <td>0.75</td><td>0.75</td><td>1.00</td><td>1.00</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d4</td><td style="color:#2980b9;"><b>Manager</b></td>
      <td>0.75</td><td>1.00</td><td>1.00</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d5</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>1.00</td><td>0.75</td><td>0.75</td><td>1.00</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d6</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>0.75</td><td>1.00</td><td>0.75</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d7</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>1.00</td><td>0.75</td><td>1.00</td><td>0.75</td>
    </tr>
    <tr style="background-color:#e8f8f5;">
      <td>d8</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.90</td><td>0.90</td><td>0.90</td><td>0.90</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td>d9</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.80</td><td>0.80</td><td>0.80</td><td>0.80</td>
    </tr>
    <tr style="background-color:#e8f8f5;">
      <td>d10</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.85</td><td>0.85</td><td>0.85</td><td>0.85</td>
    </tr>
  </tbody>
</table>


# Tasks

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#34495e; color:white;">
      <th style="padding:8px; border:1px solid #ccc;">Task</th>
      <th style="padding:8px; border:1px solid #ccc;">Type</th>
      <th style="padding:8px; border:1px solid #ccc;">Effort (h)</th>
      <th style="padding:8px; border:1px solid #ccc;">Dependencies</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#fdebd0;">
      <td>t1</td><td><b>Analysis</b></td><td>400</td><td>—</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td>t2</td><td><b>Design</b></td><td>320</td><td>t1</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td>t3</td><td><b>Design</b></td><td>240</td><td>t1</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td>t4</td><td><b>Design</b></td><td>240</td><td>t1</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td>t5</td><td><b>Implementation</b></td><td>240</td><td>t2</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td>t6</td><td><b>Implementation</b></td><td>600</td><td>t3</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td>t7</td><td><b>Implementation</b></td><td>160</td><td>t4</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td>t8</td><td><b>Testing</b></td><td>100</td><td>t5</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td>t9</td><td><b>Testing</b></td><td>80</td><td>t6</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td>t10</td><td><b>Testing</b></td><td>70</td><td>t7</td>
    </tr>
    <tr style="background-color:#f5b7b1;">
      <td>t11</td><td><b>Testing</b></td><td>80</td><td>t8, t9, t10</td>
    </tr>
  </tbody>
</table>



In [15]:
def generate_set2_data():
    # Developers (10 total) 
    devs = [
        Developer("d1", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":1.25}, "director"),
        Developer("d2", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":0.75}, "director"),
        Developer("d3", {"analysis":0.75,"design":0.75,"implementation":1.0,"testing":1.0}, "manager"),
        Developer("d4", {"analysis":0.75,"design":1.0,"implementation":1.0,"testing":0.75}, "manager"),
        Developer("d5", {"analysis":1.0,"design":0.75,"implementation":0.75,"testing":1.0}, "engineer"),
        Developer("d6", {"analysis":0.75,"design":1.0,"implementation":0.75,"testing":0.75}, "engineer"),
        Developer("d7", {"analysis":1.0,"design":0.75,"implementation":1.0,"testing":0.75}, "engineer"),
        Developer("d8", {"analysis":0.9,"design":0.9,"implementation":0.9,"testing":0.9}, "engineer"),
        Developer("d9", {"analysis":0.8,"design":0.8,"implementation":0.8,"testing":0.8}, "engineer"),
        Developer("d10",{"analysis":0.85,"design":0.85,"implementation":0.85,"testing":0.85}, "engineer"),
    ]
    # Tasks (same 11 tasks as Set1) 
    tasks = {
        "t1": Task("t1","analysis",400,[]),
        "t2": Task("t2","design",320,["t1"]),
        "t3": Task("t3","design",240,["t1"]),
        "t4": Task("t4","design",240,["t1"]),
        "t5": Task("t5","implementation",240,["t2"]),
        "t6": Task("t6","implementation",600,["t3"]),
        "t7": Task("t7","implementation",160,["t4"]),
        "t8": Task("t8","testing",100,["t5"]),
        "t9": Task("t9","testing",80,["t6"]),
        "t10": Task("t10","testing",70,["t7"]),
        "t11": Task("t11","testing",80,["t8","t9","t10"]),
    }
    return tasks, devs

### Run the Genetic Algorithm for this data set :

In [16]:
if __name__ == "__main__":
    tasks, devs = generate_set2_data()

    # CaseAll : pondérations égales
    chromA, fitA, detA = ga(tasks, devs, {"cm":.25,"ce":.25,"cc":.25,"ba":.25})

    # CaseTime : optimiser uniquement la durée (CM)
    chromT, fitT, detT = ga(tasks, devs, {"cm":1.0,"ce":0.0,"cc":0.0,"ba":0.0})

    print_results(chromA, fitA, detA, chromT, fitT, detT)

=== COMPARISON TABLE ===
Set    |  Time(All) | Time(Time) |  Multitask(All) | Multitask(Time)
Value  |     782.00 |     936.00 |          394.00 |          520.00

=== BEST SOLUTION (CaseAll) ===
Best fitness: 0.94411
Components: {'cm': 1.0, 'ce': 0.936708, 'cc': 1.0, 'ba': 0.839731}
Finish time (sim): 782
Assignments (task -> devs):
  t1 -> ['d2', 'd3', 'd6']
  t2 -> ['d8', 'd2']
  t3 -> ['d10', 'd2', 'd3']
  t4 -> ['d4', 'd6']
  t5 -> ['d9', 'd2', 'd3']
  t6 -> ['d10', 'd3']
  t7 -> ['d10', 'd7', 'd4']
  t8 -> ['d4', 'd8', 'd2']
  t9 -> ['d10', 'd9', 'd1']
  t10 -> ['d7', 'd4', 'd2']
  t11 -> ['d2', 'd1', 'd7']

=== BEST SOLUTION (CaseTime) ===
Best fitness: 1.0
Finish time (sim): 936
Assignments (task -> devs):
  t1 -> ['d2']
  t2 -> ['d2', 'd1', 'd6']
  t3 -> ['d10', 'd8']
  t4 -> ['d10', 'd5', 'd3']
  t5 -> ['d4']
  t6 -> ['d2', 'd4']
  t7 -> ['d5']
  t8 -> ['d7']
  t9 -> ['d3', 'd9', 'd4']
  t10 -> ['d6', 'd2']
  t11 -> ['d6', 'd7', 'd10']


## Test 3 : Data Set 3

## Developers

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#1f2d3d; color:white;">
      <th style="padding:8px; border:1px solid #ccc;">Developer</th>
      <th style="padding:8px; border:1px solid #ccc;">Role</th>
      <th style="padding:8px; border:1px solid #ccc;">Analysis</th>
      <th style="padding:8px; border:1px solid #ccc;">Design</th>
      <th style="padding:8px; border:1px solid #ccc;">Implementation</th>
      <th style="padding:8px; border:1px solid #ccc;">Testing</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#ecf0f1;">
      <td>d1</td><td style="color:#c0392b;"><b>Director</b></td>
      <td>1.25</td><td>1.00</td><td>1.25</td><td>1.25</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d2</td><td style="color:#c0392b;"><b>Director</b></td>
      <td>1.25</td><td>1.00</td><td>1.25</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d3</td><td style="color:#2980b9;"><b>Manager</b></td>
      <td>0.75</td><td>0.75</td><td>1.00</td><td>1.00</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d4</td><td style="color:#2980b9;"><b>Manager</b></td>
      <td>0.75</td><td>1.00</td><td>1.00</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d5</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>1.00</td><td>0.75</td><td>0.75</td><td>1.00</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d6</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>0.75</td><td>1.00</td><td>0.75</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d7</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>1.00</td><td>0.75</td><td>1.00</td><td>0.75</td>
    </tr>
    <tr style="background-color:#e8f8f5;">
      <td>d8</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.90</td><td>0.90</td><td>0.90</td><td>0.90</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td>d9</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.80</td><td>0.80</td><td>0.80</td><td>0.80</td>
    </tr>
    <tr style="background-color:#e8f8f5;">
      <td>d10</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.85</td><td>0.85</td><td>0.85</td><td>0.85</td>
    </tr>
  </tbody>
</table>


## Tasks

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#34495e; color:white;">
      <th style="padding:8px; border:1px solid #ccc;">Task</th>
      <th style="padding:8px; border:1px solid #ccc;">Type</th>
      <th style="padding:8px; border:1px solid #ccc;">Effort (h)</th>
      <th style="padding:8px; border:1px solid #ccc;">Dependencies</th>
    </tr>
  </thead>
  <tbody>
    <!-- Analysis -->
    <tr style="background-color:#fdebd0;"><td>t1</td><td><b>Analysis</b></td><td>400</td><td>—</td></tr>
    <tr style="background-color:#fdebd0;"><td>t12</td><td><b>Analysis</b></td><td>300</td><td>—</td></tr>
    <!-- Design -->
    <tr style="background-color:#d6eaf8;"><td>t2</td><td><b>Design</b></td><td>320</td><td>t1</td></tr>
    <tr style="background-color:#d6eaf8;"><td>t3</td><td><b>Design</b></td><td>240</td><td>t1</td></tr>
    <tr style="background-color:#d6eaf8;"><td>t4</td><td><b>Design</b></td><td>240</td><td>t1</td></tr>
    <tr style="background-color:#d6eaf8;"><td>t13</td><td><b>Design</b></td><td>250</td><td>t12</td></tr>
    <tr style="background-color:#d6eaf8;"><td>t16</td><td><b>Design</b></td><td>200</td><td>t12</td></tr>
    <!-- Implementation -->
    <tr style="background-color:#d5f5e3;"><td>t5</td><td><b>Implementation</b></td><td>240</td><td>t2</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t6</td><td><b>Implementation</b></td><td>600</td><td>t3</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t7</td><td><b>Implementation</b></td><td>160</td><td>t4</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t14</td><td><b>Implementation</b></td><td>400</td><td>t13</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t17</td><td><b>Implementation</b></td><td>350</td><td>t16</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t19</td><td><b>Implementation</b></td><td>300</td><td>t13</td></tr>
    <!-- Testing -->
    <tr style="background-color:#fadbd8;"><td>t8</td><td><b>Testing</b></td><td>100</td><td>t5</td></tr>
    <tr style="background-color:#fadbd8;"><td>t9</td><td><b>Testing</b></td><td>80</td><td>t6</td></tr>
    <tr style="background-color:#fadbd8;"><td>t10</td><td><b>Testing</b></td><td>70</td><td>t7</td></tr>
    <tr style="background-color:#fadbd8;"><td>t15</td><td><b>Testing</b></td><td>90</td><td>t14</td></tr>
    <tr style="background-color:#fadbd8;"><td>t18</td><td><b>Testing</b></td><td>85</td><td>t17</td></tr>
    <tr style="background-color:#fadbd8;"><td>t20</td><td><b>Testing</b></td><td>75</td><td>t19</td></tr>
    <!-- Final merge -->
    <tr style="background-color:#f5b7b1;">
      <td>t11</td><td><b>Testing</b></td><td>80</td><td>t8, t9, t10</td>
    </tr>
    <tr style="background-color:#f1948a;">
      <td>t21</td><td><b>Testing</b></td><td>95</td><td>t15, t18, t20</td>
    </tr>
  </tbody>
</table>


In [17]:
#  données de test : Set3 
def generate_set3_data():
    #  Developers 
    devs = [
        Developer("d1", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":1.25}, "director"),
        Developer("d2", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":0.75}, "director"),
        Developer("d3", {"analysis":0.75,"design":0.75,"implementation":1.0,"testing":1.0}, "manager"),
        Developer("d4", {"analysis":0.75,"design":1.0,"implementation":1.0,"testing":0.75}, "manager"),
        Developer("d5", {"analysis":1.0,"design":0.75,"implementation":0.75,"testing":1.0}, "engineer"),
        Developer("d6", {"analysis":0.75,"design":1.0,"implementation":0.75,"testing":0.75}, "engineer"),
        Developer("d7", {"analysis":1.0,"design":0.75,"implementation":1.0,"testing":0.75}, "engineer"),
        Developer("d8", {"analysis":0.9,"design":0.9,"implementation":0.9,"testing":0.9}, "engineer"),
        Developer("d9", {"analysis":0.8,"design":0.8,"implementation":0.8,"testing":0.8}, "engineer"),
        Developer("d10",{"analysis":0.85,"design":0.85,"implementation":0.85,"testing":0.85}, "engineer"),
    ]
    #  Tasks 
    tasks = {
        "t1": Task("t1","analysis",400,[]),
        "t2": Task("t2","design",320,["t1"]),
        "t3": Task("t3","design",240,["t1"]),
        "t4": Task("t4","design",240,["t1"]),
        "t5": Task("t5","implementation",240,["t2"]),
        "t6": Task("t6","implementation",600,["t3"]),
        "t7": Task("t7","implementation",160,["t4"]),
        "t8": Task("t8","testing",100,["t5"]),
        "t9": Task("t9","testing",80,["t6"]),
        "t10": Task("t10","testing",70,["t7"]),
        "t11": Task("t11","testing",80,["t8","t9","t10"]),
        "t12": Task("t12","analysis",300,[]),
        "t13": Task("t13","design",250,["t12"]),
        "t14": Task("t14","implementation",400,["t13"]),
        "t15": Task("t15","testing",90,["t14"]),
        "t16": Task("t16","design",200,["t12"]),
        "t17": Task("t17","implementation",350,["t16"]),
        "t18": Task("t18","testing",85,["t17"]),
        "t19": Task("t19","implementation",300,["t13"]),
        "t20": Task("t20","testing",75,["t19"]),
        "t21": Task("t21","testing",95,["t15","t18","t20"]),
    }
    return tasks, devs 

### Run the Genetic Algorithm for this data set :

In [18]:
if __name__ == "__main__":
    tasks, devs = generate_set3_data()

    # CaseAll : pondérations égales
    chromA, fitA, detA = ga(tasks, devs, {"cm":.25,"ce":.25,"cc":.25,"ba":.25})

    # CaseTime : optimiser uniquement la durée (CM)
    chromT, fitT, detT = ga(tasks, devs, {"cm":1.0,"ce":0.0,"cc":0.0,"ba":0.0})

    print_results(chromA, fitA, detA, chromT, fitT, detT)

=== COMPARISON TABLE ===
Set    |  Time(All) | Time(Time) |  Multitask(All) | Multitask(Time)
Value  |     893.00 |     967.00 |         1501.00 |         1095.00

=== BEST SOLUTION (CaseAll) ===
Best fitness: 0.907866
Components: {'cm': 1.0, 'ce': 0.801994, 'cc': 1.0, 'ba': 0.82947}
Finish time (sim): 893
Assignments (task -> devs):
  t1 -> ['d2', 'd3']
  t2 -> ['d6', 'd3', 'd8']
  t3 -> ['d10', 'd3']
  t4 -> ['d1', 'd3', 'd7']
  t5 -> ['d9', 'd4', 'd3']
  t6 -> ['d6', 'd1', 'd10']
  t7 -> ['d3', 'd1']
  t8 -> ['d9']
  t9 -> ['d3', 'd8', 'd1']
  t10 -> ['d9', 'd3', 'd1']
  t11 -> ['d9', 'd1', 'd3']
  t12 -> ['d7', 'd4', 'd10']
  t13 -> ['d3', 'd2', 'd10']
  t14 -> ['d2', 'd3']
  t15 -> ['d4', 'd9', 'd2']
  t16 -> ['d7', 'd9', 'd1']
  t17 -> ['d2', 'd3', 'd9']
  t18 -> ['d2']
  t19 -> ['d7', 'd8', 'd2']
  t20 -> ['d2', 'd10', 'd7']
  t21 -> ['d7', 'd4', 'd2']

=== BEST SOLUTION (CaseTime) ===
Best fitness: 1.0
Finish time (sim): 967
Assignments (task -> devs):
  t1 -> ['d7', 'd9']
  t2


### Analysis and discussion

The following table presents the results obtained for the three input sets using our genetic algorithm implementation.  
For each set, two cases are evaluated:

- **CaseAll**: all objectives are considered simultaneously (time, multitasking, continuity, and balance).
- **CaseTime**: only the project completion time is optimized.

### Note on Result in this table

the results presented in this table were obtained by running the entire code at once for each dataset in another Jupyter file. For each set, the same full code was copied into a single cell, and only the input data (Set1, Set2, or Set3) was changed. 
However, after restructuring the code into multiple Jupyter cells (with functions defined once and then called separately in the main cell), the results started to vary between executions.

### Table : Comparison of experimental results obtained with this algorithm

| Metric | Set1 (11 tasks, 7 devs) |  | Set2 (11 tasks, 10 devs) |  | Set3 (21 tasks, 10 devs) |  |
|---|---|---|---|---|---|---|
|  | **All** | **Time** | **All** | **Time** | **All** | **Time** |
| **Project duration** | 824 | 833 | 749 | 949 | 981 | 965 |
| **Multitasking time** | 111 | 741 | 438 | 533 | 1106 | 1848 |
| **Mean (#devs / effortᵢ)** | 1.94e-02 | 3.05e-02 | 2.76e-02 | 3.62e-02 | 1.12e-02 | 1.96e-02 |
| **Variance (#devs / effortᵢ)** | 1.68e-04 | 3.91e-04 | 2.74e-04 | 4.18e-04 | 4.31e-05 | 8.87e-05 |
| **Best fitness** | 0.9488 | 1.0000 | 0.9475 | 1.0000 | 0.9098 | 1.0000 |


### Mean and Variance of (#Developers / Effort)

For each task \( i \), we compute the following ratio:
$$
r_i = \frac{|\text{Developers}_i|}{\text{Effort}_i}
$$
The **mean** value over all tasks is:
$$
\mu = \frac{1}{N} \sum_{i=1}^{N} r_i
$$
The **variance** is computed as:
$$
\sigma^2 = \frac{1}{N} \sum_{i=1}^{N} (r_i - \mu)^2
$$
where \( N \) is the total number of tasks.

#### General observation
Across all sets, **CaseAll provides a better compromise** between project duration and work quality, while **CaseTime always reaches the best fitness for time but at the cost of excessive multitasking**.  
This confirms the importance of using a multi-objective approach when managing complex software projects.

# Estimating time and space complexity
### Algorithmic Time Complexity

## Overall Time Complexity
**COMPLEXITY:**  

$$
O(n^3)
$$

where $n$ is the number of tasks.

---

## Calculation Steps

### 1. Initialization — $O(n^2)$

- Building data structures: `tidx`, `didx`, dependency graph  
- Speed matrix construction `speed[n_D x n]`:

$$
O(n_D \times n) = O(n^2)
$$

- Critical path calculation (`minTS`):

$$
O(n^2)
$$

- Initial population creation:

$$
O(\text{pop} \times n)
$$

---

### 2. One Evaluation — `simulate()` → $O(n^3)$

- $n$ tasks to schedule  
- $n$ simulation ticks
- $n$ operations per tick (dependency graph traversal)

$$
n \times n \times n = n^3
$$

---

### 3. Per Generation — $O(n^3)$

- Population evaluations:

$$
\text{pop} = 20 \Rightarrow 20 \times O(n^3)
$$

- Selection + Crossover + Mutation:

$$
O(n)
$$

- Elitism sorting:

$$
O(\text{pop} \log \text{pop}) = O(1)
$$

**Dominant term:**

$$
20 \times O(n^3) = O(n^3)
$$

---

### 4. Total Genetic Algorithm — $O(n^3)$

- Number of generations:

$$
\text{gens} = 50
$$

$$
50 \times O(n^3) = O(n^3)
$$

Constants ($\text{pop}=20$, $\text{gens}=50$) disappear in Big-O notation.

---

## Final Explanation

The overall complexity is **$O(n^3)$** because, at each generation, the algorithm simulates the scheduling of $n$ tasks over approximately $n$ time steps (ticks), traversing the dependency graph at each tick.  
This simulation is repeated for each individual in the population and over multiple generations, leading to a cubic complexity in the number of tasks.


### Space Complexity

We analyze the space complexity in terms of:

$$
n = T
$$

where $T$ is the number of tasks.

---

## Assumptions

- $D$ (number of developers):  
  $$
  O(n) \quad \text{(worst case)}
  $$

- $E$ (number of dependencies):  
  $$
  O(n) \text{ to } O(n^2) \quad \text{(depending on graph density)}
  $$

- $k$ ($k_{\max}$ maximum number of developers per task):  
  $$
  3 \quad \text{(constant)}
  $$

- $P$ (population size):  
  $$
  20 \quad \text{(constant)}
  $$

---

## Memory Components

### 1. Population Storage

- $P$ chromosomes  
- $T$ tasks per chromosome  
- $k$ developers per task  

$$
P \times T \times k = 20 \times n \times 3 = O(n)
$$

---

### 2. Speed Matrix

- Stores the effective speed for each $(\text{developer}, \text{task})$ pair

$$
D \times T = n \times n = O(n^2)
$$

**← Dominant term**

---

### 3. Dependency Graph

- Adjacency lists: `prereq[T]` and `adj[T]`

$$
O(T + E)
$$

- Sparse graph:
$$
O(n)
$$

- Dense graph:
$$
O(n^2)
$$

---

### 4. Simulation Arrays

- Tracking arrays: `rem[T]`, `done[T]`, `load[D]`, etc.

$$
O(T + D) = O(n)
$$

---

## Final Result

Total space usage:

$$
O(n) + O(n^2) + O(n) + O(n)
$$

The dominant term is:

$$
O(n^2)
$$

---

## Space Complexity

$$
\boxed{\text{Space Complexity} = O(n^2)}
$$

---

## Explanation

The $O(n^2)$ space complexity is dominated by the **speed matrix** $\text{speed}[D][T]$, which stores the effective working speed of each developer on each task.  
This precomputation is essential for efficient simulation and avoids repeated calculations during the $O(n^3)$ time complexity operations.

Although the population stores $20$ complete solutions, it only contributes $O(n)$ space because $P$ is constant.  
In practice, the dependency graph usually requires $O(n)$ space since most real-world project graphs are sparse.


### Note on Result Stability and Execution Setup

It is important to note that the results presented in the comparison table were obtained by running the entire code at once for each dataset in Jupyter. For each set, the same full code was copied into a single cell, and only the input data (Set1, Set2, or Set3) was changed. Using this approach, the results were stable and reproducible across multiple executions.

However, after restructuring the code into multiple Jupyter cells (with functions defined once and then called separately in the main cell), the results started to vary between executions. This behavior is mainly due to the use of randomness in the genetic algorithm and the fact that the random seed (s) is affected by the execution order of cells in Jupyter. As a result, even with the same seed value, different execution paths can lead to different solutions.

### Experiment Analysis and Feedback on the Publication

The results obtained with this implementation are generally not as good as those reported in the original article, especially in terms of project duration and multitasking time. This difference is expected and mainly explained by two factors. First, the article does not specify the genetic algorithm parameters (population size, mutation rate, number of generations), which strongly affect solution quality. Second, the exact datasets used by the authors are not available; only the number of tasks and developers is known, while task efforts, dependencies, skills, and seniority are missing.

Despite these limitations, the same global trends are obtained. The case Time reduces project duration but increases multitasking, while the case All limits multitasking at the cost of longer project duration. Larger and complex data (like Set3) are also more difficult to optimize. So, the results is alligned with the article, even if exact numerical reproduction is not possible.